# Building models from math-spec programs

This notebook is a tour of the `linopy.spec` feature: build a full linopy model
from a **math-spec** program (a YAML description of an optimization problem)
plus a bag of data, solve it, read named results back as arrays, and round-trip
the whole thing through netCDF.

The idea in one line: **a spec is the maths, the sources are the numbers.** The
spec names dimensions, parameters, variables, constraints and an objective over
labelled axes; you supply the labels and the values separately. `linopy` binds
the two together and emits variables, constraints and an objective that align
and broadcast by dimension, exactly as if you had written them by hand.

We work through, in order:

1. Enabling v1 semantics and the `math-spec` dependency.
2. The anatomy of a spec, section by section.
3. Binding data and building a model with `Model.from_spec`.
4. Solving, and folding **named expressions** back into arrays.
5. `retain` modes and `evaluate` — what data stays on the model.
6. **Absence and coverage** — the rule that decides when a missing row is
   refused. This is the conceptual heart of the feature.
7. Lookups and grouped sums.
8. Temporal operators (`shift`).
9. Synthetic data for any spec.
10. Persistence: netCDF round-trip and `Model.copy()`.

> This notebook runs headless under `nbconvert`. It needs the `math-spec`
> package and the HiGHS solver, both pulled in by linopy's `solvers` and `spec`
> dependency groups.

In [ ]:
import math_spec
import pandas as pd
import xarray as xr
import yaml

import linopy
from linopy import Model, read_netcdf
from linopy.spec import ModelSpec, SpecDataError
from linopy.spec.testing import synthetic_sources

# A spec-built model uses linopy's v1 semantics. Set it once, up front.
linopy.options["semantics"] = "v1"

print("linopy    ", linopy.__version__)
print("math_spec ", math_spec.__version__)
print("solvers   ", linopy.available_solvers)
assert "highs" in linopy.available_solvers

## 1. A worked spec: least-cost dispatch

Here is a complete, self-contained spec. It is the classic **economic
dispatch** problem: run a fleet of generators as cheaply as possible so that
supply meets demand in every hour.

Read it top to bottom — every section is explained right after.

In [ ]:
DISPATCH = """
description: Least-cost dispatch of a generator fleet against an hourly load.

dimensions:
  snapshot: { dtype: int, description: dispatch periods }
  generator: { description: generating units }

parameters:
  p_max: { dims: [generator], description: installed capacity }
  load:  { dims: [snapshot],  description: demand to be met }
  cost:  { dims: [generator], description: marginal cost }

variables:
  p:
    description: output of a generator in a snapshot
    foreach: [snapshot, generator]
    where: "p_max > 0"
    bounds: { lower: 0, upper: p_max }

constraints:
  power_balance:
    foreach: [snapshot]
    expression: sum(p, over=generator) == load

objective:
  sense: minimize
  expression: sum(p * cost)

expressions:
  spend: sum(p * cost, over=generator)
  usage: p / p_max
"""
print(DISPATCH)

### What each section means

- **`dimensions`** — the labelled axes of the problem. Here `snapshot` (an
  integer time index) and `generator` (unit names). A dimension's `dtype`
  constrains the labels you may supply for it.
- **`parameters`** — named input data, each declared over some dimensions.
  `p_max` is one number per generator, `load` one per snapshot, `cost` one per
  generator. The spec declares the *shape*; you supply the *values* later.
- **`variables`** — the unknowns. `p` exists `foreach: [snapshot, generator]`,
  so one decision variable per (hour, unit). `where: "p_max > 0"` masks the
  variable off wherever a generator has no capacity. `bounds` fixes the feasible
  range: output is non-negative and at most the installed capacity `p_max`.
- **`constraints`** — `power_balance` holds `foreach: [snapshot]`: in every
  hour, the generators' total output must equal the load. `sum(p,
  over=generator)` collapses the generator axis, leaving one equation per
  snapshot.
- **`objective`** — minimise total spend, `sum(p * cost)` over everything.
- **`expressions`** — *named* expressions. These are **not** part of the
  optimization. They are post-solve read-outs: after solving you can ask for
  `spend` (cost per hour) or `usage` (output as a fraction of capacity) and get
  them back as numeric arrays. More on this below.

Notice there are **no numbers** in the spec, except the structural `0`. The
spec is reusable across any fleet and any set of hours.

## 2. Supplying the data

Data is a plain mapping keyed by the names the spec declares: one entry per
dimension (its labels), one per parameter (its values). linopy reads it **by
key, on demand** — it never iterates your mapping beyond the keys it needs.

Three binding rules are worth knowing, because they make the result
predictable:

1. A dimension's members come **only** from the source keyed by that
   dimension's name.
2. Their **order is your order** — linopy never sorts them.
3. A parameter source is read for **values, not labels**; it is aligned onto the
   dimension members you gave.

In [ ]:
generator = pd.Index(["wind", "gas"], name="generator")
snapshot = pd.Index([0, 1, 2], name="snapshot")

dispatch_data = {
    "snapshot": snapshot,
    "generator": generator,
    "p_max": pd.Series([100.0, 200.0], index=generator),
    "load": pd.Series([80.0, 150.0, 50.0], index=snapshot),
    "cost": pd.Series([0.0, 50.0], index=generator),  # wind free, gas costly
}
dispatch_data

## 3. Building the model

`Model.from_spec(spec, sources)` lowers the spec, binds the data and emits a
normal linopy `Model`. The `spec` argument is flexible: a path, YAML text, a
`dict`, or a `math_spec.Spec`. (A pre-lowered `Program` is refused — it has no
YAML form to keep on the model.)

`add_spec` builds into an *empty* model; `from_spec` is sugar that makes the
model for you and forwards any `Model(...)` keyword arguments.

In [ ]:
m = Model.from_spec(DISPATCH, dispatch_data)

print("variables  ", list(m.variables))
print("constraints", list(m.constraints))
print("sense      ", m.objective.sense)
m

The variable `p` is a genuine linopy variable over `(snapshot, generator)`, and
`power_balance` a genuine constraint over `snapshot`. From here everything is
ordinary linopy — you can inspect, print and manipulate them.

In [ ]:
print(m.variables["p"])

In [ ]:
print(m.constraints["power_balance"])

## 4. Solve, then fold named expressions

Solving is ordinary linopy. Wind is free, so it is used to its 100 MW cap first;
gas covers the rest. Total spend at the optimum is 2500.

In [ ]:
m.solve(solver_name="highs", output_flag=False)
print("termination:", m.termination_condition)
print("objective:  ", m.objective.value)
m.solution["p"]

### Named expressions become data, and stay maths too

`m.spec` is the accessor onto the program the model was built from. Its
`expressions` mapping returns a `NamedExpression` for each name — three views of
the same quantity:

- `.node` — the formula as math-spec's lowered expression: the symbolic handle.
- `.expression` — the **unsolved** linopy expression, variables still symbolic
  and parameters already bound. A `LinearExpression`, a bare `Variable`, an
  array, or a scalar (a named expression is affine, so never quadratic).
- `.solution` — the expression **folded** over the solution: every variable
  replaced by its solved value, every parameter by the data it was bound to, the
  arithmetic run on xarray.

`spend` = `sum(p * cost, over=generator)` folds to the cost incurred each hour;
`usage` = `p / p_max` folds to each unit's utilisation.

In [ ]:
print(repr(m.spec))
spend = m.spec.expressions["spend"]

print("\nspend.expression (unsolved linopy expression):")
print(spend.expression)

print("\nspend.solution (folded over the solution):")
print(spend.solution)

print("\nusage.solution:")
print(m.spec.expressions["usage"].solution)

### The model as maths

The accessor typesets the whole model, delegating to math-spec:
`m.spec.to_latex()`, `.to_markdown()` and `.to_typst()`. In a notebook the
accessor renders as Markdown on its own; here we show it explicitly.

In [ ]:
from IPython.display import Markdown

Markdown(m.spec.to_markdown())

A named expression that reads only data (no variables) has a `.solution`
**before** a solve too — it needs a solution only if it actually references a
variable. Subscripting an unknown name raises a `KeyError` with a suggestion
(the fold is lazy, so the error is on the subscript, not on a view).

In [ ]:
try:
    m.spec.expressions["spent"]
except KeyError as e:
    print("KeyError:", e)

## 5. `retain`: what data stays on the model

Folding needs the parameters an expression reads. `retain` controls which
parameters linopy keeps in `model.parameters` after building:

| `retain`   | keeps in `model.parameters`                     |
|------------|-------------------------------------------------|
| `"report"` | only parameters the named expressions read (default) |
| `"all"`    | every parameter                                 |
| `"none"`   | nothing                                         |

`spend` reads `cost`, `usage` reads `p_max`, neither reads `load` — so
`"report"` keeps `cost` and `p_max` but drops `load`.

In [ ]:
for retain in ["report", "all", "none"]:
    mm = Model.from_spec(DISPATCH, dispatch_data, retain=retain)
    print(f"retain={retain!r:9} -> parameters kept: {sorted(mm.parameters.data_vars)}")

### `evaluate`: fold against fresh data

With `retain="none"` nothing is kept, so `expressions[name].solution` cannot
fold. For that case (or any expression whose parameters were not retained) there
is `spec.evaluate(name, sources)`: it returns a `NamedExpression` whose
parameters are rebound from a **fresh** bag of data, folding against the model's
solution.

The catch: `evaluate` reads the solution the model already holds, so the fresh
sources must describe the **same dimension labels in the same order**.
Mislabelling a dimension is refused with a `SpecDataError`.

In [ ]:
lean = Model.from_spec(DISPATCH, dispatch_data, retain="none")
lean.solve(solver_name="highs", output_flag=False)

# .solution cannot fold: no parameters were retained.
try:
    lean.spec.expressions["spend"].solution
except SpecDataError as e:
    print("SpecDataError:", str(e)[:90], "...\n")

# evaluate rebinds from fresh sources; .solution folds:
print(lean.spec.evaluate("spend", dispatch_data).solution)

In [ ]:
# Relabelling a dimension is refused: evaluate reads the held solution.
wrong = {**dispatch_data, "generator": pd.Index(["solar", "coal"], name="generator")}
try:
    lean.spec.evaluate("spend", wrong)
except SpecDataError as e:
    print("SpecDataError:", e)

## 6. Absence and coverage — one rule, every position

This is the concept that makes spec-built models predictable on **sparse** data.
Real data has holes: a parameter table may simply not list a value for some
member. math-spec's answer is **uniform** — a missing row is **refused
wherever it is used**, no matter which position in the maths it sits in:

- **As a coefficient**, a missing row is refused. It would otherwise read as a
  silent zero and drop the term while the row stays — that's exactly the
  ambiguity the rule closes.
- **As a variable bound**, a missing row is refused. Zero is a bound, not the
  absence of one, so linopy refuses to guess.
- **As a constant side** of a constraint, a missing row is refused. It would
  bind the constraint, so it must be present.
- **As a divisor**, a missing row is refused. Zero is not a divisor.
- A shift `offset` or window `width` given by a parameter *name* is a
  coefficient too, so a hole there is refused the same way.

Crucially, each rule is checked against the rows the declaration **actually
builds** — a `where:` that removed a coordinate has already answered, so a slot
you masked off is never demanded. There is no silent zero-fill anywhere; if
zero is what you mean, you say so, either by masking the coordinate out or by
filling the data yourself.

Let's see all four positions refuse the same kind of hole.

In [ ]:
T = pd.Index([0, 1, 2], name="t")

SPARSE = {
    "dimensions": {"t": {"dtype": "int"}},
    "parameters": {"c": {"dims": ["t"]}, "w": {"dims": ["t"]}},
    "variables": {"x": {"foreach": ["t"], "bounds": {"lower": 0, "upper": 10}}},
    "constraints": {"cap": {"foreach": ["t"], "expression": "w * x <= c"}},
    "objective": {"sense": "maximize", "expression": "sum(x, over=t)"},
}

# w has no value at t=0. As the COEFFICIENT of x, the missing row would
# otherwise be read as 0 and the term dropped -- that's refused, not guessed.
w_hole = pd.Series([1.0, 1.0], index=T[1:])  # missing t=0
c_full = pd.Series([0.0, 4.0, 5.0], index=T)


def refuse(spec, data, label):
    try:
        Model.from_spec(spec, {"t": T, **data})
    except SpecDataError as e:
        print(f"[{label}]\n  {e}\n")


refuse(SPARSE, {"w": w_hole, "c": c_full}, "coefficient")

The other three positions refuse the same kind of hole, joining the
coefficient. Each `SpecDataError` names the position and how many rows are
short.

In [ ]:
c_hole = pd.Series([4.0, 5.0], index=T[1:])  # missing t=0

# (a) a hole in a variable bound
bound_spec = {
    **SPARSE,
    "variables": {"x": {"foreach": ["t"], "bounds": {"lower": 0, "upper": "c"}}},
}
refuse(bound_spec, {"w": pd.Series([1.0, 1.0, 1.0], index=T), "c": c_hole}, "bound")

# (b) a hole in a constant side (right-hand side that binds the constraint)
refuse(SPARSE, {"w": pd.Series([1.0, 1.0, 1.0], index=T), "c": c_hole}, "constant side")

# (c) a hole in a divisor
div_spec = {
    **SPARSE,
    "constraints": {"cap": {"foreach": ["t"], "expression": "x / w <= c"}},
}
refuse(div_spec, {"w": w_hole, "c": c_full}, "divisor")

Two escape hatches fix the coefficient hole above, and both build and solve.

**(a) `where:`** — the coordinate does not exist there, so there is no row to
cover. Add `where: "w"` to the `cap` constraint and t=0 drops out entirely.

**(b) Fill the data** — if zero really is what you mean, say so:
`w.fillna(0.0)` (or any dense series) supplies the row instead of leaving a
hole for linopy to guess at.

In [ ]:
where_spec = {
    **SPARSE,
    "constraints": {
        "cap": {"foreach": ["t"], "where": "w", "expression": "w * x <= c"}
    },
}
wm = Model.from_spec(where_spec, {"t": T, "w": w_hole, "c": c_full})
wm.solve(solver_name="highs", output_flag=False)
print("where: t=0 has no cap row ->", wm.objective.value)

fm2 = Model.from_spec(SPARSE, {"t": T, "w": w_hole.reindex(T).fillna(0.0), "c": c_full})
fm2.solve(solver_name="highs", output_flag=False)
print("fillna(0.0): t=0's cap is 0*x <= 0 ->", fm2.objective.value)

And the same masking escape hatch on the variable and constraint together:
`x` and its cap only exist where `live` is true, so the hole in `c` at the
masked position is fine.

In [ ]:
masked_spec = {
    **SPARSE,
    "parameters": {**SPARSE["parameters"], "live": {"dims": ["t"], "dtype": "bool"}},
    "variables": {
        "x": {"foreach": ["t"], "where": "live", "bounds": {"lower": 0, "upper": "c"}}
    },
    "constraints": {
        "cap": {"foreach": ["t"], "where": "live", "expression": "w * x <= c"}
    },
}
live = pd.Series([True, True], index=T[1:])  # off at t=0, where c is missing
mm = Model.from_spec(
    masked_spec,
    {"t": T, "w": pd.Series([1.0, 1.0, 1.0], index=T), "c": c_hole, "live": live},
)
built = int((mm.variables["x"].labels != -1).sum())
print(f"x occupies {built} of 3 slots; the masked t=0 needed no data.")

## 7. Lookups and grouped sums

A **lookup** maps each member of one dimension to a member of another — think
"which bus is this generator on". The spec declares it under `lookups:`, and an
expression can then sum a per-generator quantity **into** per-bus totals with
`sum(..., by=<lookup>)`.

In [ ]:
GROUPED = {
    "dimensions": {"generator": {}, "bus": {"dtype": "str"}},
    "lookups": {"gen_bus": {"over": "generator", "into": "bus"}},
    "parameters": {"capacity": {"dims": ["generator"]}},
    "variables": {
        "imports": {"foreach": ["bus"], "bounds": {"lower": 0, "upper": 100}}
    },
    "constraints": {
        "import_limit": {
            "foreach": ["bus"],
            "expression": "imports <= sum(capacity, by=gen_bus)",
        }
    },
    "objective": {"sense": "maximize", "expression": "sum(imports, over=bus)"},
}
gens = pd.Index(["g1", "g2"], name="generator")
grouped_data = {
    "bus": ["north", "south"],
    "generator": gens,
    "gen_bus": pd.Series(["north", "north"], index=gens),  # both gens on north
    "capacity": pd.Series([3.0, 4.0], index=gens),
}
gm = Model.from_spec(GROUPED, grouped_data)
gm.solve(solver_name="highs", output_flag=False)
print(gm.solution["imports"].to_series())
print("south has no generators -> its grouped capacity is 0, not a gap.")

Note `south` has no generators mapped to it. Its group is **empty**, and an
empty group on a constant side sums to a clean **zero**, not a missing-data gap.
An empty group is a legitimate answer; a member with no value is still refused.

## 8. Temporal operators: `shift`

For time-coupled problems the language provides operators that walk an axis:
`shift` (offset a series along a dimension), `at` (index through a lookup),
`sum_back` (a trailing window). `shift(expr, over=snapshot, offset=1,
edge='wrap')` gives "the value one step earlier, wrapping at the ends" — exactly
what a storage balance needs.

Below, a battery links consecutive hours: its state of charge equals the
previous hour's charge, plus what it stored, minus what it released. With a
cheap-then-expensive price profile, the optimizer buys extra cheap energy, banks
it, and discharges when power is dear.

In [ ]:
STORAGE = """
description: A battery shifts cheap energy into expensive hours.
dimensions:
  snapshot: { dtype: int }
parameters:
  load:    { dims: [snapshot] }
  price:   { dims: [snapshot] }
  soc_max: { dims: [] }
variables:
  gen:       { foreach: [snapshot], bounds: { lower: 0, upper: 1000 } }
  charge:    { foreach: [snapshot], bounds: { lower: 0, upper: soc_max } }
  discharge: { foreach: [snapshot], bounds: { lower: 0, upper: soc_max } }
  soc:       { foreach: [snapshot], bounds: { lower: 0, upper: soc_max } }
constraints:
  balance:
    foreach: [snapshot]
    expression: gen + discharge - charge == load
  storage:
    foreach: [snapshot]
    expression: soc == shift(soc, over=snapshot, offset=1, edge='wrap') + charge - discharge
objective:
  sense: minimize
  expression: sum(gen * price)
expressions:
  cost: sum(gen * price, over=snapshot)
"""
snap = pd.Index(range(6), name="snapshot")
storage_data = {
    "snapshot": snap,
    "load": pd.Series([10, 10, 10, 10, 10, 10], index=snap, dtype=float),
    "price": pd.Series([1, 1, 1, 9, 9, 9], index=snap, dtype=float),
    "soc_max": 20.0,
}
bm = Model.from_spec(STORAGE, storage_data, retain="all")
bm.solve(solver_name="highs", output_flag=False)
print("objective:", bm.objective.value)
print(
    pd.DataFrame(
        {
            "price": storage_data["price"],
            "gen": bm.solution["gen"].to_series(),
            "charge": bm.solution["charge"].to_series(),
            "discharge": bm.solution["discharge"].to_series(),
            "soc": bm.solution["soc"].to_series(),
        }
    ).round(1)
)

The generator over-produces while power is cheap (hour 2 runs at 30 to fill the
battery), the battery discharges through the expensive hours, and the folded
`cost` expression reports total generation spend.

In [ ]:
print("folded cost:", float(bm.spec.expressions["cost"].solution))

## 9. Synthetic data for any spec

A spec declares exactly what data it needs, which is enough to invent some. The
`synthetic_sources` helper reads a lowered program and fabricates dense data of
the right shapes — labels numbered per dimension, parameters a linear ramp. The
result builds and solves, and tells you nothing about a real system. It is what
the test suite and benchmarks use to exercise any spec.

In [ ]:
program = math_spec.to_program(yaml.safe_load(DISPATCH))
fake = synthetic_sources(program, n=4)
print("keys:", sorted(fake))
print("\ngenerated 'generator' labels:", list(fake["generator"]))
print("generated 'load':")
print(fake["load"])

fm = Model.from_spec(DISPATCH, fake, retain="all")
fm.solve(solver_name="highs", output_flag=False)
print("\nsynthetic model solves:", fm.termination_condition)

## 10. Persistence: netCDF and copy

A spec-built model round-trips through netCDF and through `Model.copy()`. The
spec travels as its **YAML text**, stored as a top-level attribute and lowered
again on read. Everything else that must survive is data: the master
coordinates, the lookups and the retained parameters.

Labels are the delicate part — a partial lookup can hold a `NaN` inside an array
of strings, and no netCDF type carries that. linopy stores lookups and
object-dtype parameters as `pandas.factorize` output (integer codes plus a
category table) and records each parameter's in-memory dtype, so the exact
dtypes come back on read on both the `netcdf4` and `scipy` engines.

In [ ]:
import os
import tempfile

from linopy.testing import assert_model_equal

m2 = Model.from_spec(DISPATCH, dispatch_data, retain="report")
m2.solve(solver_name="highs", output_flag=False)

with tempfile.TemporaryDirectory() as d:
    path = os.path.join(d, "dispatch.nc")
    m2.to_netcdf(path)
    restored = read_netcdf(path)

# the models are equal, including the spec text and the retained parameters:
assert_model_equal(m2, restored)
print("round-trip equal:", True)
print("spec text preserved:", restored.spec.text == m2.spec.text)

# and the named expressions fold identically after the round-trip:
for name in restored.spec.expressions:
    xr.testing.assert_equal(
        m2.spec.expressions[name].solution, restored.spec.expressions[name].solution
    )
    print(f"  {name}: identical")

Even a `retain="none"` model round-trips: the spec text and coordinates survive,
so after loading you can still `evaluate` against fresh data.

In [ ]:
with tempfile.TemporaryDirectory() as d:
    path = os.path.join(d, "lean.nc")
    lean.to_netcdf(path)
    lean_back = read_netcdf(path)

print("no parameters retained:", list(lean_back.parameters.data_vars) == [])
print(lean_back.spec.evaluate("spend", dispatch_data))

`Model.copy()` carries the spec too, with the accessor rebound to the copy. The
copy is a fresh, unsolved model (like any linopy copy), so solve it before
folding an expression that reads a variable — the folded result then matches the
original.

In [ ]:
clone = m2.copy()
print("copy has spec:", isinstance(clone.spec, ModelSpec))
print("copy carries a solution:", "solution" in clone.variables["p"].data)

clone.solve(solver_name="highs", output_flag=False)
xr.testing.assert_equal(
    clone.spec.expressions["spend"].solution, m2.spec.expressions["spend"].solution
)
print("after solving the copy, folded expressions match the original")

## Where the code lives, and two upstream notes

The feature is a small package, `linopy/spec/`, imported only when you call
`add_spec`/`from_spec` — `import linopy` never pulls in `math_spec`. Roughly:

- `accessor.py` — `model.spec`, the `NamedExpression` views, `evaluate`, and
  whole-model typesetting (`to_latex` / `to_markdown` / `to_typst`).
- `binder.py` — the three binding rules; data onto master coordinates.
- `builder.py` — emits variables, constraints, objective; folds expressions.
- `operators.py` — `sum`, `by=`, `shift`, `at`, `sum_back`.
- `where.py` — `where:` predicates as boolean masks.
- `coverage.py` / `terms.py` — the absence rule from section 6: a missing row
  is refused wherever it is used.
- `curves.py` — the data side of `piecewise:` blocks.
- `netcdf.py` — the factorize-based persistence from section 10.
- `nodes.py` — walks over expression nodes. One workaround lives here:
  math-spec alpha.73's `program.children()` does not descend into a `Power`
  node, so parameters hidden under `**` would be missed; `nodes.py` walks into
  the base and exponent itself.

Two upstream requests shape what the typesetting shows:
[math-spec#384](https://github.com/energy-models/math-spec/issues/384) asks for a
public hook to typeset a **single** named expression, so a `NamedExpression`
could render its own formula rather than only the whole model; and the
whole-model output currently prints the objective, constraints and variable
domains, not the named expressions themselves.

### Summary

A spec is the maths over labelled axes; the sources are the numbers. `linopy`
binds them into an ordinary model, hands each named expression back as three
views — its formula, its unsolved linopy expression and its solution — refuses a
missing parameter row wherever it is used (as a coefficient, bound, constant
side or divisor alike, with `where:` and filling the data as the escape
hatches), and round-trips the lot through netCDF by keeping the spec as text
beside factorized labels.